# RT-MLIDS — 04: Adversarial Robustness Evaluation

Black-box adversarial evasion attack evaluation using **HopSkipJump** and **ZooAttack** against XGBoost.

> This is the **principal empirical contribution** of RT-MLIDS — adversarial robustness evaluation is largely absent from comparable ML-IDS literature.

**Library:** IBM Adversarial Robustness Toolbox (ART)  
**n:** 20 test samples per attack (constrained by computational budget)  
**Model evaluated:** XGBoost (best single model: 80.41% clean accuracy)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (13, 5)

## Attack Descriptions

**HopSkipJump** is a black-box boundary-search attack that iteratively queries the model to estimate the decision boundary and craft minimal perturbations that cross it. It requires only hard-label predictions (no gradients or probabilities).

**ZooAttack** estimates gradients via finite differences (zeroth-order optimisation). It is computationally more expensive per sample, which limits its effectiveness under a fixed query budget.

## TABLE 3 — Adversarial Robustness Results

Results from `rt_mlids_experiment.py` (ART 1.14, n=20 samples per attack):

In [2]:
adv_data = {
    'Condition':       ['Clean (no attack)', 'HopSkipJump (black-box)', 'ZooAttack (black-box)'],
    'Accuracy':        [0.8000,              0.2000,                    0.6500],
    'Robustness_Drop': [None,                -0.7500,                  -0.1875],
}
df_adv = pd.DataFrame(adv_data)

print('TABLE 3: Adversarial Robustness (XGBoost, n=20 test samples per attack)')
print(f"{'Condition':<32} {'Accuracy':>10}    {'Robustness Drop'}")
print('-' * 60)
for _, row in df_adv.iterrows():
    drop = f"{row['Robustness_Drop']*100:.2f}%" if row['Robustness_Drop'] else '—'
    print(f"{row['Condition']:<32} {row['Accuracy']:>10.4f}         {drop}")

TABLE 3: Adversarial Robustness (XGBoost, n=20 test samples per attack)
Condition                        Accuracy    Robustness Drop
------------------------------------------------------------
Clean (no attack)                  0.8000               —
HopSkipJump (black-box)            0.2000            -75.00%
ZooAttack (black-box)              0.6500           -18.75%


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

conditions = ['Clean', 'HopSkipJump', 'ZooAttack']
accuracies = [0.8000, 0.2000, 0.6500]
bar_colors = ['#4CAF50', '#F44336', '#FF9800']

bars = axes[0].bar(conditions, accuracies, color=bar_colors, edgecolor='white', width=0.5)
axes[0].axhline(y=0.80, linestyle='--', color='grey', linewidth=1, label='Clean baseline (0.80)')
axes[0].set_title('XGBoost Accuracy Under Attack', fontweight='bold')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1.0)
axes[0].legend()
for bar, val in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.015,
                 f'{val:.2f}', ha='center', fontsize=12, fontweight='bold')

attacks = ['HopSkipJump', 'ZooAttack']
drops   = [75.00, 18.75]
drop_colors = ['#F44336', '#FF9800']

bars2 = axes[1].bar(attacks, drops, color=drop_colors, edgecolor='white', width=0.4)
axes[1].set_title('Robustness Drop vs Clean Baseline (%)', fontweight='bold')
axes[1].set_ylabel('Accuracy Drop (%)')
axes[1].set_ylim(0, 90)
for bar, val in zip(bars2, drops):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 1,
                 f'−{val:.2f}%', ha='center', fontsize=12, fontweight='bold', color='#B71C1C')

plt.tight_layout()
plt.savefig('../docs/fig_adversarial_robustness.png', dpi=150, bbox_inches='tight')
plt.show()

## Principal Finding

**HopSkipJump reduces XGBoost accuracy from 80% → 20% — a 75% drop.**

This is the paper's most consequential result. It demonstrates that:

1. **Ensemble ML-IDS are highly effective against passive attackers** — 80% accuracy on unseen attack variants
2. **But critically vulnerable to black-box adversarial evasion** — a sophisticated adversary with black-box API access can craft evasion payloads using only hard-label queries
3. **ZooAttack is less effective (−18.75%)** because its gradient estimation via finite differences is less precise than HopSkipJump's boundary search under the same computational budget

### Operational Implication

Production ML-IDS deployment **must** incorporate adversarial defenses — at minimum:
- Adversarial training (augmenting training data with adversarial examples)
- Detection of adversarial perturbations (statistical outlier detection on input features)
- Rate limiting and anomaly detection on the IDS API itself

## Limitations

- Sample sizes (n=20 per attack) are constrained by computational budget — future work should use n≥100 for statistical confidence
- Evaluation on NSL-KDD (1999 traffic patterns) — adversarial robustness on modern encrypted traffic benchmarks (CIC-IDS-2018) is a priority for future work
- Only XGBoost was evaluated under adversarial conditions — the stacked ensemble's robustness may differ